In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from pathlib import Path
from scipy import stats

# --------------------------
# Config
# --------------------------
hadgem_dir = Path("/Users/ewellmeyer/Documents/research/HadGEM")
weights_base = Path("/Users/ewellmeyer/Documents/research/weights")
channels = [128]

# Set specific scientific style
sns.set_context("paper", font_scale=1.4)
sns.set_style("ticks") # 'whitegrid' is okay, but 'ticks' is often cleaner for journals

# --------------------------
# Helpers (Keep your loading logic)
# --------------------------
def gn_from_ch(ch: int) -> int:
    if ch in (1, 2, 4, 6): return ch
    return 8

def load_softmax_npz(ch: int):
    gn = gn_from_ch(ch)
    p = weights_base / (f"unet_ens_HG789_PR_dPdK_Softmax_unet6SR_1x_ch{ch}_k3_128x_dPbins64_gn{gn}/softmax_ensemble_analysis_arrays.npz")
    if not p.exists(): raise FileNotFoundError(f"Missing: {p}")
    return np.load(p)

def rmse_improvement(ppe_rmse, model_rmse):
    return (ppe_rmse - model_rmse) / (ppe_rmse + 1e-12) * 100

# --------------------------
# 1. Load & Organize Data into a DataFrame
# --------------------------
# (Loading logic remains the same as your script...)
gweighted = np.load(hadgem_dir / "gaussian_weighted_analysis_memberwise_sig8000.npz")
polynomial = np.load(hadgem_dir / "polynomial_regression_analysis.npz")
gw_test_idx = gweighted["test_idx"]
poly_test_idx = polynomial["test_idx"]
softmax = {ch: load_softmax_npz(ch) for ch in channels}
sm_test_idx = softmax[channels[0]]["test_indices"]
hadgem_sm_baseline = softmax[channels[0]]["rmse_ppe"][sm_test_idx]
hadgem_sm_baseline_land = softmax[channels[0]]["rmse_ppe_land"][sm_test_idx]

# Helper to build the dataframe list
data_rows = []

def add_data(values, method, region):
    for v in values:
        data_rows.append({"RMSE Improvement (%)": v, "Method": method, "Region": region})

# Global Data
add_data(rmse_improvement(gweighted["member_rmse_simple"][gw_test_idx], gweighted["member_rmse_weighted"][gw_test_idx]), "Gaussian Weighting", "Global")
add_data(rmse_improvement(polynomial["member_rmse_mean"][poly_test_idx], polynomial["member_rmse_poly3"][poly_test_idx]), "Quadratic Reg.", "Global")
for ch in channels:
    imp = rmse_improvement(hadgem_sm_baseline, softmax[ch]["rmse_softmax_mean"][sm_test_idx])
    add_data(imp, f"NN (ch={ch})", "Global")

# Land Data
add_data(rmse_improvement(gweighted["member_rmse_simple_land"][gw_test_idx], gweighted["member_rmse_weighted_land"][gw_test_idx]), "Gaussian Weighting", "Land Only")
add_data(rmse_improvement(polynomial["member_rmse_mean_land"][poly_test_idx], polynomial["member_rmse_poly3_land"][poly_test_idx]), "Quadratic Reg.", "Land Only")
for ch in channels:
    imp = rmse_improvement(hadgem_sm_baseline_land, softmax[ch]["rmse_softmax_mean_land"][sm_test_idx])
    add_data(imp, f"NN (ch={ch})", "Land Only")

# Create DataFrame
df = pd.DataFrame(data_rows)

# --------------------------
# 2. Plotting
# --------------------------
# Define Palette for consistency
method_colors = {
    "Gaussian Weighting": "#E69F00", # Colorblind friendly orange
    "Quadratic Reg.": "#56B4E9",     # Colorblind friendly blue
}
# Add NN colors dynamically
nn_palette = sns.color_palette("viridis", n_colors=len(channels))
for i, ch in enumerate(channels):
    method_colors[f"NN (ch={ch})"] = nn_palette[i]

fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)

for i, region in enumerate(["Global", "Land Only"]):
    ax = axes[i]
    subset = df[df["Region"] == region]
    
    # Main KDE Plot
    sns.kdeplot(
        data=subset, 
        x="RMSE Improvement (%)", 
        hue="Method", 
        palette=method_colors,
        fill=True, 
        alpha=0.15,       # Lighter fill to see overlaps
        linewidth=2,      # Thicker lines for the main curve
        common_norm=False,
        legend=(i==1),    # Only show legend on the second plot to save space
        ax=ax
    )
    ax.set_xlim(-10, 55)

    # Add Median Lines (Scientific touch)
    for method in subset["Method"].unique():
        median_val = subset[subset["Method"] == method]["RMSE Improvement (%)"].median()
        color = method_colors[method]
        # Draw a small vertical dash for the median
        ax.axvline(median_val, color=color, linestyle="--", alpha=0.8, linewidth=1.5, ymax=0.05)
        # Optional: Add text for median
        ax.text(median_val, ax.get_ylim()[1]*0.9, f'{median_val:.1f}', color=color, ha='center', fontsize=9)

    # Zero line (Baseline)
    ax.axvline(0, color=".3", linestyle="-", linewidth=1)
    
    # Set a and  b before title 
    fig_letters = ['A', 'B']

    ax.set_title(f"{fig_letters[i]}. {region}", fontweight="bold", pad=15)
    ax.set_xlabel("RMSE Improvement (%)")
    ax.set_ylim(0, 0.16)
    if i == 0:
        ax.set_ylabel("Density")
    else:
        ax.set_ylabel("")
        # Clean up legend
        sns.move_legend(ax, "upper right", bbox_to_anchor=(1, 1), frameon=False, title=None)

sns.despine(trim=True) # Removes top/right spines and detaches them slightly
plt.tight_layout()
plt.show()

# --------------------------
# Additional Figure: Global Only
# --------------------------
fig, ax = plt.subplots(figsize=(7, 5))

subset_global = df[df["Region"] == "Global"]

sns.kdeplot(
    data=subset_global, 
    x="RMSE Improvement (%)", 
    hue="Method", 
    palette=method_colors,
    fill=True, 
    alpha=0.15,
    linewidth=2,
    common_norm=False,
    legend=True,
    ax=ax
)
ax.set_xlim(-10, 50)

# Add Median Lines
for method in subset_global["Method"].unique():
    median_val = subset_global[subset_global["Method"] == method]["RMSE Improvement (%)"].median()
    color = method_colors[method]
    ax.axvline(median_val, color=color, linestyle="--", alpha=0.8, linewidth=1.5, ymax=0.05)

ax.axvline(0, color=".3", linestyle="-", linewidth=1)
# ax.set_title("A. Global Only", fontweight="bold", pad=15)
ax.set_xlabel("RMSE Improvement (%)")
ax.set_ylabel("Density")
sns.move_legend(ax, "upper right", frameon=False, title=None)

sns.despine(trim=True)
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 6))
sns.violinplot(
    data=df, 
    x="RMSE Improvement (%)", 
    y="Method", 
    hue="Region", 
    split=True,       # Combines Global/Land into one violin
    inner="quartile", # Shows the statistical quartiles inside
    palette="muted"
)
plt.xlim(-20, 60)
sns.despine(left=True)
plt.axvline(0, color="k", linestyle="--", alpha=0.5)
plt.show()


# --------------------------
# FIGURE 3: Combined KDE Plot (Global on top, Land Only mirrored below)
# --------------------------
from scipy import stats

fig, ax = plt.subplots(figsize=(12, 6))

# Create KDE for each method and region
x_range = np.linspace(-10, 60, 500)

for method in ["Gaussian Weighting", "Quadratic Reg."] + [f"NN (ch={ch})" for ch in channels]:
    color = method_colors[method]
    
    # Global data - plot positive density
    global_data = df[(df["Method"] == method) & (df["Region"] == "Global")]["RMSE Improvement (%)"].values
    if len(global_data) > 1:
        kde_global = stats.gaussian_kde(global_data)
        density_global = kde_global(x_range)
        ax.fill_between(x_range, 0, density_global, color=color, alpha=0.25, label=f"{method}")
        ax.plot(x_range, density_global, color=color, linewidth=2, alpha=0.8)
        
        # Add median line for global
        median_global = np.median(global_data)
        max_density_global = float(kde_global(median_global))
        ax.plot([median_global, median_global], [0, max_density_global * 0.05], 
                color=color, linestyle='--', linewidth=1.5, alpha=0.8)
    
    # Land Only data - plot negative density (flipped)
    land_data = df[(df["Method"] == method) & (df["Region"] == "Land Only")]["RMSE Improvement (%)"].values
    if len(land_data) > 1:
        kde_land = stats.gaussian_kde(land_data)
        density_land = kde_land(x_range)
        ax.fill_between(x_range, 0, -density_land, color=color, alpha=0.25)
        ax.plot(x_range, -density_land, color=color, linewidth=2, alpha=0.8)
        
        # Add median line for land
        median_land = np.median(land_data)
        max_density_land = float(kde_land(median_land))
        ax.plot([median_land, median_land], [0, -max_density_land * 0.05], 
                color=color, linestyle='--', linewidth=1.5, alpha=0.8)

# Styling
ax.axhline(0, color='k', linestyle='-', linewidth=1.5)
ax.axvline(0, color='.3', linestyle='-', linewidth=1)
ax.set_xlim(-10, 55)
ax.set_xlabel("RMSE Improvement (%)", fontsize=14, fontweight='bold')
ax.set_ylabel("Density", fontsize=14, fontweight='bold')

# Remove y-axis tick labels but keep grid
ax.set_yticks([])

# Add region labels
y_lim = ax.get_ylim()
ax.text(50, y_lim[1] * 0.9, 'A. Global', 
        fontsize=12, fontweight='bold', va='top', ha='right',
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.8, edgecolor='gray'))
ax.text(50, y_lim[0] * 0.9, 'B. Land Only', 
        fontsize=12, fontweight='bold', va='bottom', ha='right',
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.8, edgecolor='gray'))

# Legend
ax.legend(loc='upper center', frameon=False, fontsize=11)

sns.despine(left=True)
plt.tight_layout()
plt.show()